# Crawler Milho CEPEA ESALQ/B3 (Google Colab)

Este notebook adapta o `crawler.py` para execucao no Google Colab.

## Fluxo
1. Instalar dependencias
2. Fazer upload do XLS base (opcional, mas recomendado)
3. Scrapar dados diarios do CEPEA
4. Inicializar ou atualizar `cotacao_milho.xlsx`
5. Fazer download do arquivo final

In [ ]:
# Dependencias para Colab
!pip -q install curl_cffi beautifulsoup4 pandas openpyxl xlrd ipywidgets

In [ ]:
import os
import pandas as pd
import ipywidgets as widgets
from bs4 import BeautifulSoup
from curl_cffi import requests as cf
from IPython.display import display, clear_output

CEPEA_URL = 'https://cepea.org.br/br/indicador/milho.aspx'
XLS_BASE = '/content/cepea_base.xls'
XLS_SAIDA = '/content/cotacao_milho.xlsx'

print('Ambiente pronto.')

## Upload do XLS base (opcional)
Se voce tiver o XLS historico baixado do CEPEA, faca upload aqui.

Se nao enviar, o notebook continua e cria a aba `Media_Anual` vazia.

In [ ]:
from google.colab import files

print('Selecione o XLS base do CEPEA (opcional).')
uploaded = files.upload()

if len(uploaded) > 0:
    nome_arquivo = next(iter(uploaded.keys()))
    origem = f'/content/{nome_arquivo}'
    if origem != XLS_BASE:
        import shutil
        shutil.copy(origem, XLS_BASE)
    print(f'XLS base salvo como: {XLS_BASE}')
else:
    print('Nenhum XLS base enviado. Continuando sem medias anuais.')

In [ ]:
def limpar_numero(serie: pd.Series) -> pd.Series:
    # Converte padrao brasileiro (1.234,56 e 0,59%) para float
    return pd.to_numeric(
        serie.astype(str)
             .str.replace(r'\s', '', regex=True)
             .str.replace('%', '', regex=False)
             .str.replace('.', '', regex=False)
             .str.replace(',', '.', regex=False)
             .str.replace(r'[^\d.\-]', '', regex=True),
        errors='coerce',
    )


def scrape_tabela() -> pd.DataFrame:
    print(f'Conectando: {CEPEA_URL}')
    resp = cf.get(CEPEA_URL, impersonate='chrome120', timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, 'html.parser')
    tabela = soup.find('table')
    if tabela is None:
        raise RuntimeError('Tabela nao encontrada na pagina do CEPEA.')

    linhas = []
    for tr in tabela.find_all('tr'):
        cols = [td.get_text(strip=True) for td in tr.find_all('td')]
        if len(cols) == 5 and cols[0]:
            linhas.append(cols)

    if not linhas:
        raise RuntimeError('Nenhuma linha de dados encontrada na tabela.')

    df = pd.DataFrame(
        linhas,
        columns=['data_str', 'valor_rs_str', 'var_dia_str', 'var_mes_str', 'valor_usd_str'],
    )

    resultado = pd.DataFrame()
    resultado['data'] = pd.to_datetime(df['data_str'], dayfirst=True, errors='coerce')
    resultado['valor_rs'] = limpar_numero(df['valor_rs_str'])
    resultado['var_dia_pct'] = limpar_numero(df['var_dia_str'])
    resultado['var_mes_pct'] = limpar_numero(df['var_mes_str'])
    resultado['valor_usd'] = limpar_numero(df['valor_usd_str'])

    return (
        resultado
        .dropna(subset=['data', 'valor_rs'])
        .sort_values('data')
        .reset_index(drop=True)
    )


def importar_base_xls() -> pd.DataFrame:
    if not os.path.exists(XLS_BASE):
        print('[AVISO] XLS base nao encontrado. Media_Anual sera criada vazia.')
        return pd.DataFrame(columns=['ano', 'valor_rs_medio', 'valor_usd_medio'])

    raw = pd.read_excel(XLS_BASE, header=None, engine='xlrd')

    header_row = 3
    for i, row in raw.iterrows():
        if any('data' in str(c).lower() for c in row.values):
            header_row = int(i)
            break

    df = pd.read_excel(XLS_BASE, header=header_row, engine='xlrd')
    df.columns = [str(c).strip() for c in df.columns]

    def achar(palavras):
        for p in palavras:
            for col in df.columns:
                if p.lower() in col.lower():
                    return col
        return None

    col_ano = achar(['data', 'ano'])
    col_rs = achar(['r$', 'vista r', 'reais'])
    col_usd = achar(['us$', 'dolar', 'dollar'])

    resultado = pd.DataFrame()
    resultado['ano'] = df[col_ano].astype(str).str.strip() if col_ano else '?'
    resultado['valor_rs_medio'] = limpar_numero(df[col_rs]) if col_rs else float('nan')
    resultado['valor_usd_medio'] = limpar_numero(df[col_usd]) if col_usd else float('nan')

    return resultado.dropna(subset=['valor_rs_medio']).reset_index(drop=True)


def ler_diario_existente() -> pd.DataFrame:
    colunas = ['data', 'valor_rs', 'var_dia_pct', 'var_mes_pct', 'valor_usd']
    if not os.path.exists(XLS_SAIDA):
        return pd.DataFrame(columns=colunas)
    try:
        df = pd.read_excel(XLS_SAIDA, sheet_name='Diario', engine='openpyxl')
        df['data'] = pd.to_datetime(df['data'], errors='coerce')
        return df
    except Exception:
        return pd.DataFrame(columns=colunas)


def salvar_xls(df_diario: pd.DataFrame, df_anual: pd.DataFrame) -> None:
    with pd.ExcelWriter(
        XLS_SAIDA,
        engine='openpyxl',
        date_format='DD/MM/YYYY',
        datetime_format='DD/MM/YYYY',
    ) as writer:
        df_diario.to_excel(writer, sheet_name='Diario', index=False)
        df_anual.to_excel(writer, sheet_name='Media_Anual', index=False)

## Execucao interativa com widgets
Use os controles abaixo para escolher o modo e executar o crawler.

- `Atualizar`: adiciona apenas datas novas ao `cotacao_milho.xlsx`
- `Reimportar`: recria o arquivo com base no XLS historico + scraping

In [ ]:
def executar_crawler(reimportar=False):
    if reimportar or not os.path.exists(XLS_SAIDA):
        print('--- Inicializando cotacao_milho.xlsx ---')
        df_anual = importar_base_xls()
        print(f'Media_Anual: {len(df_anual)} linha(s)')

        df_diario = scrape_tabela()
        print(
            f'Diario: {len(df_diario)} linha(s) '
            f'({df_diario["data"].min().date()} -> {df_diario["data"].max().date()})'
        )

        salvar_xls(df_diario, df_anual)
        print(f'Arquivo criado: {XLS_SAIDA}')
        return

    print('--- Atualizando cotacao_milho.xlsx ---')
    df_existente = ler_diario_existente()

    if df_existente.empty:
        print('Aba Diario vazia ou ausente. Recriando arquivo...')
        df_anual = importar_base_xls()
        df_diario = scrape_tabela()
        salvar_xls(df_diario, df_anual)
        return

    datas_existentes = set(df_existente['data'].dt.normalize())
    ultima_data = df_existente['data'].max()
    print(f'Linhas atuais: {len(df_existente)} | ultima data: {ultima_data.date()}')

    df_novo = scrape_tabela()
    df_novo_filtrado = df_novo[
        ~df_novo['data'].dt.normalize().isin(datas_existentes)
    ].copy()

    if df_novo_filtrado.empty:
        print('Nenhuma data nova encontrada. Arquivo ja atualizado.')
        return

    try:
        df_anual = pd.read_excel(XLS_SAIDA, sheet_name='Media_Anual', engine='openpyxl')
    except Exception:
        df_anual = importar_base_xls()

    df_combinado = (
        pd.concat([df_existente, df_novo_filtrado], ignore_index=True)
        .drop_duplicates(subset=['data'])
        .sort_values('data')
        .reset_index(drop=True)
    )
    salvar_xls(df_combinado, df_anual)
    print(f'+{len(df_novo_filtrado)} linha(s) adicionada(s).')
    print(f'Total Diario: {len(df_combinado)}')


modo_widget = widgets.Dropdown(
    options=[('Atualizar (incremental)', False), ('Reimportar (recriar arquivo)', True)],
    value=False,
    description='Modo:',
)
botao_widget = widgets.Button(description='Executar crawler', button_style='success')
saida_widget = widgets.Output()

def _ao_clicar(_):
    with saida_widget:
        clear_output(wait=True)
        try:
            executar_crawler(reimportar=modo_widget.value)
            print('Execucao finalizada com sucesso.')
        except Exception as exc:
            print(f'Erro durante execucao: {exc}')

botao_widget.on_click(_ao_clicar)
display(widgets.VBox([modo_widget, botao_widget, saida_widget]))

In [ ]:
# Visualizacao rapida
df_diario_final = pd.read_excel(XLS_SAIDA, sheet_name='Diario', engine='openpyxl')
df_anual_final = pd.read_excel(XLS_SAIDA, sheet_name='Media_Anual', engine='openpyxl')

print('Diario - ultimas 5 linhas:')
display(df_diario_final.tail(5))

print('Media_Anual - ultimas 5 linhas:')
display(df_anual_final.tail(5))

In [ ]:
# Download do resultado
from google.colab import files
files.download(XLS_SAIDA)